# Clean public-domain-poetry
Steps:
1. remove empty and very short poems
2. remove prose
3. remove translations
4. fix the OCR slip "0" → "O"
5. clean up author names
6. remove duplicate poems

Output: `data/pdp_clean.parquet`

In [1]:
import json
import re
import pandas as pd

DATA = "../data/"

with open(DATA + "public-domain-poetry/poems.json", encoding="utf-8") as f:
    poems = pd.DataFrame(json.load(f))

poems["text"] = poems["text"].fillna("")
print(len(poems), "poems")

38499 poems


## 1. Remove empty and very short poems
We keep poems with at least 4 lines.

The plan also said to drop poems over 200 lines, but that would remove 43% of all lines, and almost
all of them are real long poems (*Hiawatha*, *Lalla Rookh*, ...), not junk. So there's no upper limit;
long poems get split into chunks later.

In [2]:
def count_lines(text):
    return len([line for line in text.split("\n") if line.strip()])


poems["n_lines"] = poems["text"].map(count_lines)

before = len(poems)
poems = poems[poems["n_lines"] >= 4]
print("removed", before - len(poems), "poems with fewer than 4 lines")

removed 804 poems with fewer than 4 lines


## 2. Remove prose
Some entries are prose (fables, short stories) with a whole paragraph on each "line".
We drop a poem if more than a quarter of its lines are over 200 characters, or if the title marks it
as prose, like "(Prose)", "- Prose" or "Poems In Prose".
(A simple "average line length > 100" rule also removed 9 Walt Whitman poems, and just searching titles for
"prose" also caught poems like "The Garden of Proserpine", so we don't use those.)

In [3]:
def share_of_long_lines(text):
    lines = [line for line in text.split("\n") if line.strip()]
    return sum(len(line) > 200 for line in lines) / len(lines)


long_lines = poems["text"].map(share_of_long_lines) > 0.25
prose_title = poems["Title"].str.contains(r"\(prose|- prose|in prose", case=False)
is_prose = long_lines | prose_title

print("removed", is_prose.sum(), "prose entries")
print(poems[is_prose]["Author"].value_counts().head(8))
poems = poems[~is_prose]

removed 227 prose entries
Author
Washington Irving                 52
Jean de La Fontaine               45
Paul Cameron Brown                22
Charles Baudelaire                15
Ivan Sergeyevich Turgenev         15
Isabella Augusta, Lady Gregory    14
John Hartley                      13
Michael Fairless                   9
Name: count, dtype: int64


## 3. Remove translations
Translated poems show the translator's style, not the poet's. This list comes from going through all
author names by hand.

In [4]:
TRANSLATED = [
    "Jean de La Fontaine", "Francesco Petrarca (Petrarch)", "Johann Wolfgang von Goethe",
    "Friedrich Schiller", "Charles Baudelaire", "Victor-Marie Hugo",
    "Michelangelo di Lodovico Buonarroti Simoni", "Bj'rnstjerne Martinius Bj'rnson",
    "Edward Powys Mathers (As Translator)", "Dante Alighieri", "Alfred Lichtenstein",
    "Ivan Sergeyevich Turgenev", "Yehuda Amichai", "Morris Rosenfeld", "Nizar Qabbani",
    "Giacomo Leopardi", "Fernando Ant'nio Nogueira Pessoa", "Bertolt Brecht", "Anna Akhmatova",
    "Heinrich Hoffmann", "Jan Kochanowski", "Adam Bernard Mickiewicz", "Aurelius Clemens Prudentius",
    "Joseph Victor von Scheffel", "Sophocles", "Andrejs Pumpurs", "Abraham a Sancta-Clara",
    "Tomas de Iriarte y Oropesa", "Aristophanes", "Ant'nio Corr'a", "Isabella Augusta, Lady Gregory",
]

before = len(poems)
poems = poems[~poems["Author"].isin(TRANSLATED)]
print("removed", before - len(poems), "translated poems")

removed 2638 translated poems


## 4. Fix the OCR slip
The scan sometimes reads "O" as "0", e.g. "0 dear old year". We replace a "0" that stands on its own
(not part of a number) with "O".

A few lines also end in a stray "0" ("they sting; 0"). That's a scanning mark, not an "O", so we delete
those first.

In [5]:
END_ZERO = r"(?<!\d)[ \xa0]*\b0[ \xa0]*$"
print(poems["text"].str.count(END_ZERO, flags=re.M).sum(), "lines end in a stray 0")
poems["text"] = poems["text"].str.replace(END_ZERO, "", regex=True, flags=re.M)

ZERO = r"(?<![\d.,])\b0\b(?![.,]?\d)"

has_zero = poems["text"].str.contains(ZERO)
print(has_zero.sum(), "poems have the slip, e.g.:")
print(re.search(".{0,20}" + ZERO + ".{0,20}", poems[has_zero]["text"].iloc[0]).group(0))

poems["text"] = poems["text"].str.replace(ZERO, "O", regex=True)
print("left after fixing:", poems["text"].str.contains(ZERO).sum())

20 lines end in a stray 0


59 poems have the slip, e.g.:
0 Nymph! with cheeks 


left after fixing: 0


## 5. Clean up author names
Remove extra bits like "(Sir)" or ", Sir". A few names are fixed by hand: shorter names that match the
Gutenberg corpus, two poets who appear under two names each, and the second James Thomson, who would
otherwise be merged with the first. Poems by "Unknown" stay in: they're used for the `Author: Unknown` header.

In [6]:
AUTHOR_MAP = {
    "Emily Elizabeth Dickinson": "Emily Dickinson",
    "Christina Georgina Rossetti": "Christina Rossetti",
    "Alice Christiana Gertrude Thompson Meynell": "Alice Meynell",
    "Alice Christiana Thompson Meynell": "Alice Meynell",
    "John Alexander McCrae": "John McCrae",
    "James Thomson -  (Bysshe Vanolis)": "James Thomson (B.V.)",  # not the same poet as "James Thomson"
}


def clean_author(name):
    if name in AUTHOR_MAP:
        return AUTHOR_MAP[name]
    name = re.sub(r"\(.*?\)", "", name)  # "Walter Scott (Sir)" -> "Walter Scott"
    return " ".join(name.replace(", Sir", "").split()).strip(" ,-")


poems["author"] = poems["Author"].map(clean_author)

changed = poems[poems["author"] != poems["Author"]][["Author", "author"]].drop_duplicates()
print(len(changed), "author names changed, e.g.:")
changed.head(10)

27 author names changed, e.g.:


,Author,author
10,Bliss Carman (William),Bliss Carman
85,D. H. Lawrence (David Herbert Richards),D. H. Lawrence
98,Emily Elizabeth Dickinson,Emily Dickinson
104,Christina Georgina Rossetti,Christina Rossetti
106,Sophie M. (Almon) Hensley,Sophie M. Hensley
121,Banjo Paterson (Andrew Barton),Banjo Paterson
165,William Arthur Dunkerley (John Oxenham),William Arthur Dunkerley
304,Alice Christiana Thompson Meynell,Alice Meynell
330,Alice Christiana Gertrude Thompson Meynell,Alice Meynell
346,Alfred Joyce Kilmer (Joyce),Alfred Joyce Kilmer


## 6. Remove duplicate poems
Some poems are in the dataset more than once (Madison Cawein alone has about 200 extra copies).
Duplicates would end up in both the training and the test set, so we keep only the first copy.
Two poems count as the same if their letters match (ignoring case, spaces and punctuation).

In [7]:
letters_only = poems["text"].str.lower().str.replace(r"[^a-z]", "", regex=True)

before = len(poems)
poems = poems[~letters_only.duplicated()]
print("removed", before - len(poems), "duplicate poems")

removed 585 duplicate poems


## Save

In [8]:
poems["n_lines"] = poems["text"].map(count_lines)  # count again, step 4 changed some lines
clean = poems[["author", "Title", "text", "n_lines"]].rename(columns={"Title": "title"})
clean.to_parquet(DATA + "pdp_clean.parquet")

print(len(clean), "poems,", clean["n_lines"].sum(), "lines,", clean["author"].nunique(), "authors")

34245 poems, 2010728 lines, 462 authors


### Candidate authors after cleaning

In [9]:
candidates = ["Emily Dickinson", "Christina Rossetti", "Edgar Allan Poe", "John Keats", "Walt Whitman", "John Milton"]
clean[clean["author"].isin(candidates)].groupby("author").agg(poems=("text", "size"), lines=("n_lines", "sum"))

,poems,lines
author,,
Christina Rossetti,239,11842
Edgar Allan Poe,79,5755
Emily Dickinson,448,5457
John Keats,145,13130
John Milton,115,19744
Walt Whitman,309,10836
